# India Data Atlas — Comprehensive Data Science & ML Research Notebook
### Executive Civic Telemetry, Socio-Economic Clustering, and Time-Series Forecasting to 2030

**Author:** India Data Atlas Research Team  
**Platform:** Python 3, Pandas, Scikit-Learn, Plotly & Seaborn  
**Dataset:** Multi-Year District Observational Catalog (2015–2024, 11 Locked Indicators, 75 UP Districts & Pan-India Hubs)

---
### Research Agenda:
1. **Data Ingestion & Validation:** High-performance tabular transformation from the atlas data service.
2. **Exploratory Data Analysis (EDA):** Statistical distributions, dispersion metrics, and outlier detection.
3. **Cross-Sectoral Correlation Analysis:** Measuring systemic inter-dependencies across literacy, child mortality, electrification, and water infrastructure.
4. **Unsupervised Socio-Economic Clustering:** K-Means clustering (k=4) & Principal Component Analysis (PCA) 2D projection.
5. **Time-Series Forecasting (2025–2030):** Ridge regression trend projections with 95% statistical confidence intervals and CAGR.
6. **Vulnerability & Poverty Index (VPI):** Composite multi-criteria decision matrix and quadrant mapping.
7. **Algorithmic Policy Simulator:** Capital budget allocation modeling and cross-sector multiplier evaluation.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic styling
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = 'Inter'
plt.rcParams['axes.edgecolor'] = '#D4AF37'

# Import atlas core engine
from core.config import INDICATORS, ALL_DISTRICTS, UP_DISTRICTS, YEARS, DEFAULT_YEAR
import core.data_service as data_service
import core.ml_engine as ml_engine

print('[+] Atlas Core Modules Loaded Successfully.')
print(f'[+] Monitored Districts: {len(ALL_DISTRICTS)} | Core Indicators: {len(INDICATORS)} | Years: {YEARS[0]}–{YEARS[-1]}')

## 1. Data Ingestion & Transformation

We ingest the complete observational dataset using `core.data_service.get_complete_dataset_dataframe()`.  
This generates a tidy, long-form pandas DataFrame containing observations for all districts, indicators, metrics, and observation years.

In [ ]:
# Load full dataset into a Pandas DataFrame
df = data_service.get_complete_dataset_dataframe('uttar-pradesh')
print(f'DataFrame Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('\n--- Sample Observational Records ---')
df.head(10)

## 2. Descriptive Statistical Analysis & Data Hygiene

Examining metric ranges, null distributions (simulated missing observations), and statistical summaries.

In [ ]:
# Check data completeness & status breakdown
print('Data Status Distribution:')
print(df['status'].value_counts(normalize=True).round(3) * 100)

print('\n--- Summary Statistics for Key Indicators (2024) ---')
pivot_2024 = df[df['year'] == 2024].pivot_table(
    index=['district_name', 'division'],
    columns='metric_key',
    values='value'
)
pivot_2024[['literacy_rate', 'infant_mortality_rate', 'piped_water_coverage', 'household_electrification', 'total_crime_cases']].describe().round(2)

## 3. Cross-Sectoral Correlation Analysis

Understanding how foundational civic infrastructure interacts: does higher piped water coverage correlate with lower infant mortality? How does rural road connectivity impact education and electricity access?

In [ ]:
# Compute correlation matrix across key development indicators
core_metrics = [
    'literacy_rate',
    'school_enrollment',
    'infant_mortality_rate',
    'institutional_deliveries',
    'piped_water_coverage',
    'household_electrification',
    'villages_connected',
    'agricultural_productivity',
    'crime_rate'
]

corr_data = pivot_2024[core_metrics].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(
    corr_data,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    linewidths=0.5
)
plt.title('Inter-Sectoral Socio-Economic Correlation Matrix (Uttar Pradesh 2024)', fontsize=14, pad=15)
plt.show()

## 4. Unsupervised Socio-Economic Clustering (K-Means & PCA)

We apply unsupervised machine learning to classify the 75 districts of Uttar Pradesh into 4 developmental tiers:
- **Tier 1:** High Infrastructure & Advanced Human Capital
- **Tier 2:** Fast-Growing Emerging Regional Hubs
- **Tier 3:** Moderate Agrarian Plains
- **Tier 4:** High Vulnerability Priority Development Zones

We reduce dimensionality via Principal Component Analysis (PCA) for 2D latent space visualization.

In [ ]:
cluster_results = ml_engine.cluster_districts('uttar-pradesh', 2024, n_clusters=4)
cluster_df = pd.DataFrame(cluster_results['districts'])

print(f"Variance Explained: PC1 = {cluster_results['variance_explained'][0]}%, PC2 = {cluster_results['variance_explained'][1]}%")
print('\nDistrict Count per Cluster Tier:')
for c in cluster_results['clusters_metadata']:
    print(f"  * {c['name']}: {c['count']} districts")

# Plot 2D PCA Cluster Map
plt.figure(figsize=(12, 7))
palette = {c['id']: c['color'] for c in cluster_results['clusters_metadata']}

sns.scatterplot(
    data=cluster_df,
    x='pca_x',
    y='pca_y',
    hue='clusterId',
    palette=palette,
    s=120,
    alpha=0.9,
    edgecolor='#FFFFFF',
    linewidth=1.2
)

# Label key districts (Lucknow, Varanasi, Agra, Kanpur, Shravasti)
annot_districts = ['lucknow', 'varanasi', 'agra', 'kanpur-nagar', 'shravasti', 'prayagraj', 'meerut']
for _, row in cluster_df.iterrows():
    if row['districtId'] in annot_districts:
        plt.text(
            row['pca_x'] + 0.1,
            row['pca_y'] + 0.1,
            row['districtName'],
            fontsize=10,
            fontweight='bold',
            color='#1E293B'
        )

plt.title('District Developmental Tiers in 2D Latent Space (PCA + K-Means)', fontsize=14, pad=15)
plt.xlabel('Principal Component 1 (Primary Development Axis)')
plt.ylabel('Principal Component 2 (Secondary Structural Variance)')
plt.legend(title='Development Tier', loc='lower right')
plt.show()

## 5. Supervised Time-Series Forecasting (2025–2030)

Forecasting multi-year district developmental trajectories using Ridge Regression with quadratic drift terms and 95% statistical confidence envelopes.

In [ ]:
district_id = 'lucknow'
metric = 'literacy_rate'
forecast = ml_engine.predict_metric_future(district_id, 'education', metric, 2030)

hist_df = pd.DataFrame(forecast['historical'])
proj_df = pd.DataFrame(forecast['projections'])

plt.figure(figsize=(12, 6))
# Historical observation line
plt.plot(hist_df['year'], hist_df['value'], marker='o', linewidth=2.5, color='#D4AF37', label='Observed (2015–2024)')

# Forecast projection line
proj_x = [hist_df['year'].iloc[-1]] + list(proj_df['year'])
proj_y = [hist_df['value'].iloc[-1]] + list(proj_df['predicted'])
plt.plot(proj_x, proj_y, linestyle='--', marker='s', linewidth=2.5, color='#10B981', label='Ridge Forecast (2025–2030)')

# 95% Confidence Interval
ci_x = list(proj_df['year'])
plt.fill_between(ci_x, proj_df['lower_bound'], proj_df['upper_bound'], color='#10B981', alpha=0.15, label='95% Confidence Envelope')

plt.title(f'15-Year Trajectory & 2030 Projections: Literacy Rate in {district_id.title()}', fontsize=14, pad=15)
plt.xlabel('Year')
plt.ylabel('Literacy Rate (%)')
plt.ylim(50, 105)
plt.legend(loc='lower right')
plt.show()

print(f"Historical CAGR (2015–2024): {forecast['historicalCagr']}%/year")
print(f"Projected CAGR (2024–2030):  {forecast['projectedCagr']}%/year")
print(f"Model R² Goodness-of-Fit:   {forecast['r2_score']}")
print(f"2030 Projected Target:      {proj_df.iloc[-1]['predicted']}% (Bounds: [{proj_df.iloc[-1]['lower_bound']}% – {proj_df.iloc[-1]['upper_bound']}%])")

## 6. Multi-Criteria Vulnerability & Poverty Index (VPI) Matrix

We construct a composite multi-indicator vulnerability index across Education, Health, Water, Electricity, and Roads.
Districts are categorized into 4 strategic governance quadrants.

In [ ]:
vpi_res = ml_engine.compute_vpi_matrix('uttar-pradesh', 2024)
vpi_df = pd.DataFrame(vpi_res['districts'])

print('Top 5 Frontrunner Districts (Lowest Vulnerability):')
print(vpi_df[['rank', 'districtName', 'composite', 'quadrant']].head(5).to_string(index=False))

print('\nBottom 5 Priority Districts (Highest Distress):')
print(vpi_df[['rank', 'districtName', 'composite', 'quadrant']].tail(5).to_string(index=False))

# Plot Quadrants Distribution
plt.figure(figsize=(10, 5))
quad_counts = vpi_df['quadrant'].value_counts()
colors = ['#10B981', '#D4AF37', '#3B82F6', '#EF4444']
quad_counts.plot(kind='bar', color=colors, edgecolor='#07090E', linewidth=1.5)
plt.title('District Distribution Across Governance Quadrants (VPI 2024)', fontsize=14, pad=15)
plt.xlabel('Quadrant')
plt.ylabel('Number of Districts')
plt.xticks(rotation=0)
plt.show()

## 7. Algorithmic Policy Intervention Simulator

Simulating the socio-economic return on investment (ROI) of targeted capital allocations.
Models diminishing marginal returns and cross-sector multiplier effects.

In [ ]:
# Run simulation for a district with customized budget
sim_output = ml_engine.simulate_policy_intervention(
    district_id='lucknow',
    budget_health_cr=200.0,
    budget_education_cr=160.0,
    budget_infra_cr=250.0,
    budget_water_cr=100.0,
    objective='integrated-welfare'
)

print(sim_output['dossierText'])

## 8. Exporting Analytical Outputs for Academic Thesis

Exporting analytical tables, cluster assignments, and forecasts to CSV for inclusion in project deliverables.

In [ ]:
# Export clusters and VPI rankings
os.makedirs('exports', exist_ok=True)
vpi_df.to_csv('exports/uttar_pradesh_district_vpi_rankings_2024.csv', index=False)
cluster_df.to_csv('exports/uttar_pradesh_district_clusters_2024.csv', index=False)

print('[+] Datasets successfully exported to exports/ directory:')
print('  - exports/uttar_pradesh_district_vpi_rankings_2024.csv')
print('  - exports/uttar_pradesh_district_clusters_2024.csv')
print('\n[+] India Data Atlas Research Notebook Complete.')